# Books to Scrape — Web Scraping Project

Scraped book information from the Books to Scrape website using Python, Requests, and BeautifulSoup.

The project extracts book titles and prices from 50 pages and organizes the collected data into a Pandas DataFrame.

## Project Overview

### Objective

Collect book information from all 50 pages of the Books to Scrape website.

### Data collected

- Book title
- Book price

### Tools used

- Python
- Requests
- BeautifulSoup
- Pandas

### Output

The scraped data is stored in a Pandas DataFrame and exported to an Excel file.

## 1. Scrape Book Data

In [36]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

all_books = []

for page in range(1, 51):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    r = requests.get(url, timeout=10)
    r.raise_for_status()
    soup = BeautifulSoup(r.content, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.find("h3").find("a")["title"]
        price = book.find("p", class_="price_color").get_text(strip=True)

        all_books.append({
            "title": title,
            "price": price
        })


## 2. Create DataFrame

In [37]:
df = pd.DataFrame(all_books)
df.head()

,title,price
0,A Light in the Attic,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10
3,Sharp Objects,£47.82
4,Sapiens: A Brief History of Humankind,£54.23


## 3. Validate the Scraped Data

### Basic Checks

In [38]:
df.shape

(1000, 2)

In [39]:
df.isnull().sum()

,0
title,0
price,0


In [40]:
df["title"].duplicated().sum()

np.int64(1)

In [41]:
df[df["title"].duplicated()]

,title,price
358,The Star-Touched Queen,£32.30


In [42]:
df[df["title"] == "The Star-Touched Queen"]

,title,price
236,The Star-Touched Queen,£46.02
358,The Star-Touched Queen,£32.30


> Note: Duplicate titles were found, but the records were retained because the same title appears with different prices in the source website.

In [43]:
df['price'] = df['price'].str.replace('£', '')
df.head()

,title,price
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sharp Objects,47.82
4,Sapiens: A Brief History of Humankind,54.23


In [44]:
df['price'] = df['price'].astype(float)
df.rename(columns={"price": "price_gbp"}, inplace=True)

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      1000 non-null   object 
 1   price_gbp  1000 non-null   float64
dtypes: float64(1), object(1)
memory usage: 15.8+ KB


## 4. Export Data

In [46]:
df.to_excel("books_data.xlsx", index = False)